# Lab 10 Challenge: worked solution

The challenge: split the policies into paragraphs that keep their source, measure hit@1 and hit@3 again on the lab's
ten labelled questions, then make the answer cite its chunks and check the citations in code. This notebook runs on
its own: it repeats the config cell, the embedding call and the labelled questions from the lab, stores the 20
policies twice (whole files, as in the lab, and paragraphs), and compares them.

## Setup: the pieces from the lab

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import OpenAI, omit
import chromadb

client = OpenAI()
EMBEDDING_MODEL = settings["EMBEDDING_MODEL"]
CORPUS = ROOT / "data" / "corpus"
policy_files = sorted(CORPUS.glob("policy_*.md"))
print(len(policy_files), "policies")


def embed_all(texts):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]

20 policies


`embed_all` is the lab's batch call from Part B: one call, one vector per text, in the same order. The labelled
questions are the lab's Part G, unchanged:

In [3]:
LABELLED = [
    ("How long do I have to return an unopened CD?", "policy_returns.md"),
    ("When is support staffed?", "policy_support_hours.md"),
    ("Do gift cards expire?", "policy_gift_cards.md"),
    ("How many business days does a refund take?", "policy_refunds.md"),
    ("My download link expired. Can I get a new one?", "policy_digital_downloads.md"),
    ("Can I pay with bitcoin?", "policy_payment.md"),
    ("Is there a phone number I can call for support?", "policy_data_retention.md"),
    ("Can you ship the second disc of my set separately?", "policy_shipping.md"),
    ("What is Chinook's revenue target for 2014?", None),
    ("What is the CEO's personal mobile number?", None),
]
question_vectors = embed_all([question for question, _ in LABELLED])

We embed the ten questions once and reuse their vectors for both collections, so the two searches differ only in
what was stored.

## Task 1: Split the policies into paragraphs and measure again

### The whole files, as in the lab

One record per policy file, the file name as the id and in the metadata.

In [4]:
chroma = chromadb.EphemeralClient()

whole = chroma.get_or_create_collection("whole_files")
whole_texts = [p.read_text() for p in policy_files]
whole.add(ids=[p.name for p in policy_files], documents=whole_texts,
          metadatas=[{"file": p.name} for p in policy_files], embeddings=embed_all(whole_texts))
print("whole files:", whole.count())

whole files: 20


### Paragraph chunks that keep their source

Each policy's text splits on blank lines (`"\n\n"`) into paragraphs; the first is the `# ` heading line, which we
skip as a chunk of its own and put in front of every other paragraph instead. Without it, a paragraph such as *"They
do not expire."* would lose its subject, just as the size split lost the damaged-twice rule's heading in Part C.

In [5]:
chunks = []
for path in policy_files:
    text = path.read_text().strip()
    title = text.splitlines()[0][2:]
    paragraphs = [block.strip() for block in text.split("\n\n")[1:] if block.strip()]
    for number, paragraph in enumerate(paragraphs):
        chunks.append({
            "id": f"{path.stem}#{number}",
            "file": path.name,
            "text": f"{title}: {paragraph}",
        })

print(len(chunks), "paragraph chunks")
for chunk in chunks:
    if chunk["file"] == "policy_shipping.md":
        print(chunk["id"], "|", chunk["text"])

67 paragraph chunks
policy_shipping#0 | Shipping: Domestic (same country as the billing address): 3 to 7 business days.
policy_shipping#1 | Shipping: International: 10 to 21 business days. Customs fees are the customer's.
policy_shipping#2 | Shipping: We ship physical orders once. We do not split a single invoice across two parcels.
policy_shipping#3 | Shipping: Free shipping on physical orders over 50 dollars before tax.


The shipping policy becomes four chunks, one rule each; `policy_shipping#2` holds the rule about splitting an invoice
across parcels, with its heading in front. The id says the file and the paragraph; the metadata keeps the file name,
which the measure below reads.

In [6]:
paragraphs = chroma.get_or_create_collection("paragraphs")
paragraphs.add(ids=[c["id"] for c in chunks], documents=[c["text"] for c in chunks],
               metadatas=[{"file": c["file"]} for c in chunks], embeddings=embed_all([c["text"] for c in chunks]))
print("paragraphs:", paragraphs.count())

paragraphs: 67


### hit@1 and hit@3 at file level

A hit counts when a record from the labelled file is in the top `k`, whichever paragraph it is. `file_rank` reads the
file names from the metadata of the top three results and returns the position of the labelled file (1 for first),
or `None`.

In [7]:
def file_rank(collection, vector, label, k=3):
    got = collection.query(query_embeddings=[vector], n_results=k)
    result_files = [metadata["file"] for metadata in got["metadatas"][0]]
    return result_files.index(label) + 1 if label in result_files else None


ranks = []
print(f"{'question':52} {'whole':>6} {'paragraphs':>11}")
for (question, label), vector in zip(LABELLED, question_vectors):
    if label is None:
        continue
    pair = (file_rank(whole, vector, label), file_rank(paragraphs, vector, label))
    ranks.append(pair)
    print(f"{question:52} {str(pair[0] or '-'):>6} {str(pair[1] or '-'):>11}")

for column, name in [(0, "whole files"), (1, "paragraphs")]:
    column_ranks = [pair[column] for pair in ranks]
    hit1 = sum(rank == 1 for rank in column_ranks)
    hit3 = sum(rank is not None for rank in column_ranks)
    print(f"{name:12} hit@1 {hit1}/{len(column_ranks)} | hit@3 {hit3}/{len(column_ranks)}")

question                                              whole  paragraphs
How long do I have to return an unopened CD?              1           1
When is support staffed?                                  2           1
Do gift cards expire?                                     1           1
How many business days does a refund take?                1           1
My download link expired. Can I get a new one?            1           1
Can I pay with bitcoin?                                   1           1
Is there a phone number I can call for support?           -           2
Can you ship the second disc of my set separately?        -           1
whole files  hit@1 5/8 | hit@3 6/8
paragraphs   hit@1 7/8 | hit@3 8/8


The paragraphs raise hit@1 from 5 of 8 to 7 of 8 and hit@3 from 6 of 8 to 8 of 8. The two questions the whole files missed at `k=3` are now found (the second disc first, the phone question second), and the staffing question moves from second to first. The whole-files column is the lab's policies-only column, with the same 5 and 6.

Which paragraphs came first, and what happened to the two unanswerable questions? The best cosine is $1 - d^2/2$, as
in the lab.

In [8]:
for (question, label), vector in zip(LABELLED, question_vectors):
    if label in (None, "policy_data_retention.md", "policy_shipping.md", "policy_support_hours.md"):
        got = paragraphs.query(query_embeddings=[vector], n_results=3)
        best_whole = whole.query(query_embeddings=[vector], n_results=1)
        print(question)
        print(f"   whole files: best cosine {1 - best_whole['distances'][0][0] / 2:.4f} {best_whole['ids'][0][0]}")
        for chunk_id, distance in zip(got["ids"][0], got["distances"][0]):
            print(f"   paragraph:   cosine {1 - distance / 2:.4f} {chunk_id}")

When is support staffed?
   whole files: best cosine 0.5274 policy_support_reps.md
   paragraph:   cosine 0.5368 policy_support_hours#0
   paragraph:   cosine 0.5101 policy_support_reps#3
   paragraph:   cosine 0.4837 policy_support_reps#0
Is there a phone number I can call for support?
   whole files: best cosine 0.4146 policy_support_hours.md
   paragraph:   cosine 0.4324 policy_returns#3
   paragraph:   cosine 0.4229 policy_data_retention#4
   paragraph:   cosine 0.4008 policy_support_hours#0
Can you ship the second disc of my set separately?
   whole files: best cosine 0.4406 policy_damaged_media.md
   paragraph:   cosine 0.4680 policy_shipping#2
   paragraph:   cosine 0.4488 policy_refunds#1
   paragraph:   cosine 0.4221 policy_damaged_media#1
What is Chinook's revenue target for 2014?
   whole files: best cosine 0.2036 policy_escalations.md
   paragraph:   cosine 0.2199 policy_escalations#1
   paragraph:   cosine 0.1839 policy_returns#3
   paragraph:   cosine 0.1784 policy_preord

- **Staffing:** the hours sentence alone, `policy_support_hours#0` (0.5368), now beats the best representatives
  paragraph (0.5101). As a whole file, the hours shared its vector with three other rules and lost to the
  representatives file (0.5274).
- **Phone:** `policy_returns#3` ("Start a return by writing to support ...") comes first at 0.4324, and
  `policy_data_retention#4` ("... This shop's support is email only.") second at 0.4229: found at `k=3`, not at `k=1`.
- **Second disc:** `policy_shipping#2`, the one-parcel rule, is first at 0.4680.
- **Unanswerable:** the revenue target stays low (0.2036 for the best whole file, 0.2199 for the best paragraph). The
  CEO question jumps from 0.3039 to 0.4675: `policy_escalations#3` is the single sentence about the manager's phone
  number, and on its own it looks very much like the question. It now scores level with the shipping rule that does
  answer its question (0.4680). Smaller chunks sharpen the search for answers and for near misses alike.

## Task 2: Cite the chunks, and check the citations in code

The lab's rules plus one line: end with `Sources:` and the ids used, or `Sources: none`. Each chunk is packed under its
id in square brackets, so the ids the model may cite are the ones it can see.

In [9]:
RULES = ("Answer the question using only the documents below. "
         "If they do not contain the answer, say you do not know. "
         "End with one line 'Sources:' followed by the ids, in square brackets, of the documents you used, "
         "or 'Sources: none'.")


def answer_with_sources(question, k=3):
    got = paragraphs.query(query_embeddings=[embed_all([question])[0]], n_results=k)
    retrieved = got["ids"][0]
    packed = "\n\n".join(f"[{chunk_id}]\n{text}" for chunk_id, text in zip(retrieved, got["documents"][0]))
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": RULES},
            {"role": "user", "content": f"Documents:\n\n{packed}\n\nQuestion: {question}"},
        ],
        max_completion_tokens=200,
        reasoning_effort=EFFORT or omit,
    )
    return retrieved, response.choices[0].message.content

The check takes the text after the last `Sources:` in the reply (`rsplit("Sources:", 1)` splits once, from the
right; the model does not always put it on a line of its own), finds every id between square brackets with a regular
expression (`\[([^\]]+)\]`: an opening bracket, then everything up to the closing one), and tests each against the
retrieved ids. A reply with no `Sources:` at all is reported as such, not passed. A pass means every cited chunk was really given to the model; it does not mean the chunk supports the
sentence, which still needs reading.

In [10]:
import re


def check_sources(reply, retrieved):
    if "Sources:" not in reply:
        return None, None
    cited = re.findall(r"\[([^\]]+)\]", reply.rsplit("Sources:", 1)[1])
    unknown = [chunk_id for chunk_id in cited if chunk_id not in retrieved]
    return cited, unknown


for question in ["Can you ship the second disc of my set separately?",
                 "When is support staffed?",
                 "What is the CEO's personal mobile number?"]:
    retrieved, reply = answer_with_sources(question)
    cited, unknown = check_sources(reply, retrieved)
    mark = "✅" if cited is not None and not unknown else "❌"
    print(question)
    print("   retrieved:", retrieved)
    print("   reply:    ", reply.replace("\n", " | "))
    print(f"   {mark} cited {cited}, not retrieved: {unknown}")

Can you ship the second disc of my set separately?
   retrieved: ['policy_shipping#2', 'policy_refunds#1', 'policy_damaged_media#1']
   reply:     No. We don’t split a single invoice across separate parcels.   | Sources: [policy_shipping#2]
   ✅ cited ['policy_shipping#2'], not retrieved: []


When is support staffed?
   retrieved: ['policy_support_hours#0', 'policy_support_reps#3', 'policy_support_reps#0']
   reply:     Support is staffed Monday to Friday, 09:00–17:00 in the shop’s local time, except on public holidays. |  | Sources: [policy_support_hours#0]
   ✅ cited ['policy_support_hours#0'], not retrieved: []


What is the CEO's personal mobile number?
   retrieved: ['policy_escalations#3', 'policy_support_reps#2', 'policy_data_retention#4']
   reply:     I don’t know. The documents don’t provide the CEO’s personal mobile number. Sources: [policy_escalations#3]
   ✅ cited ['policy_escalations#3'], not retrieved: []


- **Second disc:** "No. We don't split a single invoice across separate parcels.", citing `[policy_shipping#2]`, the
  paragraph that says so.
- **Staffing:** the hours, citing `[policy_support_hours#0]`.
- **CEO:** "I don't know", and it still cites `[policy_escalations#3]`, the sentence about the manager's phone number.

All three citations were among the retrieved chunks. In the CEO reply, `Sources:` came on the same line as the
answer, which is why the check reads after the last `Sources:` instead of looking for a line of its own. The check
confirms that each cited chunk was given to the model; whether the chunk supports the sentence still takes reading.
Here each cited chunk fits its reply.

**What the two tasks show together.** Paragraph chunks that keep their heading found the one-sentence answers the whole files missed (hit@3 8 of 8 against 6 of 8), and their ids make citations precise enough to check in code. The same precision lifted the CEO question's best score to the level of a real answer, so the refusal still has to come from the generate step.